# Tender RAG - the pipeline, one step at a time

This notebook runs our RAG pipeline stage by stage and prints what comes out of
each stage, so you can see the whole process instead of just the final answer.

The business problem: a government tender runs to 20-30 pages of dense clauses.
A bidder has to find things like the earnest money deposit, the closing date and
the estimated cost. Missing one of them can get the bid rejected. We built a
small RAG system that answers those questions from the actual tender PDF and
shows which page the answer came from.

The six steps, one file each, in the `rag/` folder:

| Step | File | What it does |
|---|---|---|
| 1 | `rag/step1_load.py` | read the PDF, clean the text |
| 2 | `rag/step2_chunk.py` | cut the text into overlapping chunks |
| 3 | `rag/step3_embed_store.py` | turn chunks into vectors, save to Chroma |
| 4 | `rag/step4_retrieve.py` | find the chunks closest to the question |
| 5 | `rag/step5_prompt.py` | put those chunks into the prompt |
| 6 | `rag/step6_generate.py` | call the LLM, with a LiteLLM fallback |

## Setup

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

# This notebook lives in notebooks/, but the rag package and the data folder are
# one level up. Work from the project root so the imports and paths line up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
# sentence-transformers prints a loading bar every time it loads the model;
# this keeps the notebook output readable.
os.environ["TQDM_DISABLE"] = "1"

print("working from:", PROJECT_ROOT)

load_dotenv(override=True)

if "GROQ_API_KEY" in os.environ:
    print("Groq API key loaded, begins with", os.environ["GROQ_API_KEY"][:4])
else:
    print("GROQ_API_KEY not found. Copy .env.example to .env and put your key in it.")

working from: /Users/nishkarsh/Projects/tender-rag
Groq API key loaded, begins with gsk_


---
## Step 1: load the tender PDF and clean the text

We use `PyPDFLoader`, which gives one LangChain `Document` per page. Keeping
pages separate matters, because the page number travels with the text all the
way to the final answer and becomes the citation.

In [2]:
from rag.step1_load import load_all_tenders, load_tender_pdf

pages = load_all_tenders()

print(f"\nTotal pages with text: {len(pages)}")
print(f"Total characters      : {sum(len(p.page_content) for p in pages)}")

  loaded iitpkd_lift.pdf: 22 pages with text


  loaded niti_sharp_tender.pdf: 16 pages with text


  loaded nitt_glove_box.pdf: 10 pages with text

Total pages with text: 48
Total characters      : 99975


### What does one page look like after cleaning?

The raw text from a PDF is messy. Some of these tenders put every single word on
its own line, because each word is positioned separately in a table. If we left
that alone, a chunk would be a column of words instead of a sentence, and
retrieval would get much worse. `clean_text()` detects that and reflows it.

In [3]:
first_page = pages[0]
print("metadata:", first_page.metadata)
print()
print(first_page.page_content[:700])

metadata: {'tender': 'iitpkd_lift', 'page': 1}

NOTICE INVITING TENDER FOR NAME OF WORK: SITC of 4 Passenger Home Lift for Director Bungalow (V01) in Sahyadri Campus Sl. No. Events Date and Time 1 Notice Inviting BID (NIT) No. 17/IITPKD/EWD/ELE/2026-27/036 2 Date of Publication 28-04-2026 3 Date / Time of closing 11-05-2026, 1100 hrs 4 Opening of Bids 11-05-2026, 1130 hrs 5 Estimated Cost put to bid Rs. 28,00,000/- 6 Earnest Money Deposit (EMD) Rs. 56,000/- Engineering works Division Indian Institute of Technology Palakkad (Nila Campus) Near Gramalakshmi Mudralayam, Kanjikode West, Palakkad – 678 623- Email: ewd@iitpkd.ac.in


### Before and after cleaning

Here is the same page straight from the loader, with no cleaning, so you can see
exactly what `clean_text()` fixes.

In [4]:
from langchain_community.document_loaders import PyPDFLoader
from rag.step1_load import clean_text

raw_page = PyPDFLoader("data/tenders/iitpkd_lift.pdf").load()[0].page_content

print("--- RAW (first 200 characters, shown with the newlines visible) ---")
print(repr(raw_page[:200]))
print()
print("--- CLEANED ---")
print(clean_text(raw_page)[:400])

--- RAW (first 200 characters, shown with the newlines visible) ---
'NOTICE\n \nINVITING\n \nTENDER\n \nFOR\n \nNAME\n \nOF\n \nWORK:\n   \nSITC\n \nof\n \n4\n \nPassenger\n \nHome\n \nLift\n \nfor\n \nDirector\n \nBungalow\n \n(V01)\n \nin\n \nSahyadri\n \nCampus\n \n \nSl.\n \nNo.\n \nEvents\n \nDate\n \nand\n \nTime'

--- CLEANED ---
NOTICE INVITING TENDER FOR NAME OF WORK: SITC of 4 Passenger Home Lift for Director Bungalow (V01) in Sahyadri Campus Sl. No. Events Date and Time 1 Notice Inviting BID (NIT) No. 17/IITPKD/EWD/ELE/2026-27/036 2 Date of Publication 28-04-2026 3 Date / Time of closing 11-05-2026, 1100 hrs 4 Opening of Bids 11-05-2026, 1130 hrs 5 Estimated Cost put to bid Rs. 28,00,000/- 6 Earnest Money Deposit (EMD)


---
## Step 2: split the pages into chunks

Why chunk at all? Two reasons.

1. A tender is far too long to fit in the model's context window.
2. Even if it fitted, it would hurt. The one line that answers the question
   would be buried in thousands of irrelevant lines.

So we cut the document into small overlapping pieces and search over those.

`RecursiveCharacterTextSplitter` tries paragraph breaks first, then single
newlines, then spaces, then characters. It only moves to a smaller separator
when a piece is still too big, which keeps related text together.

In [5]:
from rag.step2_chunk import CHUNK_OVERLAP, CHUNK_SIZE, split_into_chunks

chunks = split_into_chunks(pages)

print(f"chunk_size={CHUNK_SIZE}, chunk_overlap={CHUNK_OVERLAP}")
print(f"{len(pages)} pages  ->  {len(chunks)} chunks")

sizes = [len(c.page_content) for c in chunks]
print(f"chunk length: smallest {min(sizes)}, largest {max(sizes)}, average {sum(sizes)//len(sizes)}")

chunk_size=300, chunk_overlap=60
48 pages  ->  441 chunks
chunk length: smallest 1, largest 300, average 268


In [6]:
# Look at three chunks in a row, to see the overlap between them.
for c in chunks[10:13]:
    print(f"--- {c.metadata['tender']} page {c.metadata['page']} "
          f"(start_index {c.metadata['start_index']}) ---")
    print(c.page_content)
    print()

--- iitpkd_lift page 2 (start_index 1710) ---
participate in more than one bid, the bids (of both the individual and the partnership/consortium/joint venture) are liable to be rejected. 1.7. The bidder shall bear all costs associated with the preparation and submission of his bid and IITPKD shall in no case be responsible or liable for those

--- iitpkd_lift page 2 (start_index 1949) ---
IITPKD shall in no case be responsible or liable for those costs, regardless of the conduct or outcome of the tender process. 1.8. The Tender Document is not transferable. The bidder shall make a copy of the tender document before submitting the same to the concerned oﬃce. 1.9. IITPKD will respond

--- iitpkd_lift page 2 (start_index 2191) ---
the same to the concerned oﬃce. 1.9. IITPKD will respond to any request for clariﬁcation or modiﬁcation of the Tender Document that is received up to FIVE (05) days prior to the deadline for submission of bids prescribed by IITPKD. For this purpose, the prospecti

### How we chose the chunk size

We did not just take the default. We started at `chunk_size=1000`, which is the
value used in the class RAG notebook, and the results were poor.

The reason is that page 1 of a tender is a summary table holding many different
facts at once: the NIT number, the dates, the estimated cost, the EMD. At 1000
characters that whole table becomes a single chunk, so its embedding is an
average of many topics and it does not match any one question strongly.

To check this properly we wrote down six questions whose answers we had looked
up by hand in the PDFs, and counted how often the expected figure appeared
anywhere in the retrieved chunks. The cell below re-runs that test.

In [7]:
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

from rag.step3_embed_store import get_embedding_model

# question -> the string we verified by hand in the PDF
CHECKS = [
    ("What is the earnest money deposit for the home lift?", "56,000"),
    ("What is the estimated cost put to bid for the lift?",  "28,00,000"),
    ("What is the last date and time for closing of bids?",  "11-05-2026"),
    ("What is the EMD amount for the glove box tender?",     "6000"),
    ("What is the cost of the tender document?",             "158"),
    ("What is the bid validity period?",                     "75"),
]

embeddings = get_embedding_model()

print(f"{'chunk_size':>11} {'chunks':>8} {'found':>8}")
print("-" * 30)
for size, overlap in [(300, 60), (500, 100), (800, 160), (1000, 200), (1500, 300)]:
    test_chunks = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap, add_start_index=True
    ).split_documents(pages)

    test_store = Chroma.from_documents(test_chunks, embeddings,
                                       collection_name=f"test{size}")
    found = sum(
        any(expected in d.page_content for d in test_store.similarity_search(q, k=4))
        for q, expected in CHECKS
    )
    print(f"{size:>11} {len(test_chunks):>8} {found:>6}/{len(CHECKS)}")
    test_store.delete_collection()

/Users/nishkarsh/Projects/tender-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 chunk_size   chunks    found
------------------------------


        300      441      5/6


        500      275      3/6


        800      171      3/6


       1000      142      1/6


       1500      100      2/6


300 characters was clearly the best, so that is what `step2_chunk.py` uses.

This is the kind of thing the class notebook means when it writes
`# hyperparameter that can be tuned` next to these values.

---
## Step 3: embeddings and the vector store

An embedding model turns a piece of text into a list of numbers that represents
its meaning. Texts that mean similar things get vectors pointing in similar
directions, so we can compare meaning by comparing numbers.

This is what lets the system answer "what is the EMD?" when the tender actually
says "earnest money deposit" - a keyword search would find nothing there.

In [8]:
from rag.step3_embed_store import EMBEDDING_MODEL, get_embedding_model

print("embedding model:", EMBEDDING_MODEL)

embeddings = get_embedding_model()
vector = embeddings.embed_query("earnest money deposit")

print("one embedding is a list of", len(vector), "numbers")
print("first 8 numbers:", [round(v, 4) for v in vector[:8]])

embedding model: sentence-transformers/all-MiniLM-L6-v2
one embedding is a list of 384 numbers
first 8 numbers: [-0.014, 0.0784, -0.0612, -0.0001, -0.0337, 0.0015, 0.0333, -0.0206]


### Checking that similar meanings really do give similar vectors

Cosine similarity is the cosine of the angle between two vectors. It runs from
-1 (opposite) to 1 (same direction). Chroma uses this idea internally, but it is
worth computing once by hand to see that it behaves the way we claim.

In [9]:
import numpy as np


def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


def similarity(text1, text2):
    return cosine_similarity(embeddings.embed_query(text1),
                             embeddings.embed_query(text2))


pairs = [
    ("last date for submission of bids",
     "Date and time of closing of bids is 11-05-2026"),
    ("last date for submission of bids",
     "The glove box must have an oxygen sensor."),
    ("earnest money deposit",
     "EMD amount payable by the bidder"),
]

for first, second in pairs:
    print(f"{similarity(first, second):+.4f}   {first[:34]:36} | {second[:48]}")

+0.7260   last date for submission of bids     | Date and time of closing of bids is 11-05-2026
-0.0148   last date for submission of bids     | The glove box must have an oxygen sensor.
+0.2582   earnest money deposit                | EMD amount payable by the bidder


The first pair scores **0.73**, even though the two sentences share almost no
words - "last date for submission" and "closing of bids" mean the same thing.
The second pair, about a completely different topic, scores near zero. That gap
is the whole basis of retrieval.

The third pair is worth noticing too. "earnest money deposit" against "EMD"
scores only about **0.26** - much lower than we expected. A small embedding
model does not reliably know that the abbreviation and the full phrase are the
same thing. It still beats the unrelated sentence, which is why retrieval works,
but it is a real weakness and it is the reason hybrid retrieval (BM25 together
with semantic search) is on our improvements list: BM25 would match the literal
string "EMD" exactly.

### Store the chunks

Chroma keeps the vector, the chunk text and the metadata together in one
collection, and writes it to `chroma_db/`. We only pay the embedding cost once;
after this, the app starts instantly.

In [10]:
from rag.step3_embed_store import build_vector_store

store = build_vector_store(chunks)
print("chunks stored:", len(store.get()["ids"]))

chunks stored: 441


---
## Step 4: retrieval

This is the "Retrieval" in Retrieval Augmented Generation.

The question goes through the same embedding model, and Chroma compares that one
vector against all the stored chunk vectors and returns the closest ones. Only
those chunks go into the prompt.

The score shown below is a distance, so **lower means closer in meaning**.

In [11]:
from rag.step4_retrieve import TOP_K, retrieve_with_scores

question = "What is the earnest money deposit for the home lift?"
print("Question:", question)
print("TOP_K =", TOP_K)
print()

for i, (chunk, score) in enumerate(retrieve_with_scores(question, k=4), start=1):
    print(f"--- chunk {i} | {chunk.metadata['tender']} page {chunk.metadata['page']} "
          f"| distance {score:.4f} ---")
    print(chunk.page_content[:250])
    print()

Question: What is the earnest money deposit for the home lift?
TOP_K = 8

--- chunk 1 | iitpkd_lift page 2 | distance 0.8764 ---
or explanation by any other employee of any of the Sections/Departments of the Institute, shall be taken to bind or fetter the Institute. 1.11. EARNEST MONEY DEPOSIT (EMD) 1.12. The bidder shall furnish EMD of Rs.56,000/- (Rupees Fifty Six Thousand O

--- chunk 2 | niti_sharp_tender page 1 | distance 0.9018 ---
75 davs from the Bid End Date Bid Opening Venue Committee Room No.136, NITI Bhawan Earnest Money Deposit Rs.20,000/- fRupees Twenty Thousand 0nlyJ EMD Validity Period 120 days from the Bid End Date EMD Submission Last Date 7l/09 /2020 up ro 04:00 PM 

--- chunk 3 | niti_sharp_tender page 3 | distance 0.9060 ---
(The name ofthe document) [Copies of ITR of last three years) Details of Earnest Money Deposit (Rs.20,000/-): (a) Banker's Cheque No./Demand Draft No./FDR (bl Date (c) Name of Issuing Bank Annexure-l B Signatures of the apolicant bidder 5.

--- 

### Metadata filtering

We indexed three different tenders, and a question like "what is the EMD?"
matches all three, because all three talk about EMD. Each chunk carries its
tender name in its metadata, so we can restrict the search to one document while
still searching by meaning.

Notice how the sources change when the filter is applied.

In [12]:
from rag.step4_retrieve import retrieve_chunks

for tender in [None, "iitpkd_lift", "nitt_glove_box"]:
    found = retrieve_chunks("What is the earnest money deposit?", k=4, tender=tender)
    sources = sorted({f"{d.metadata['tender']} p{d.metadata['page']}" for d in found})
    print(f"filter={str(tender):18} ->  {', '.join(sources)}")

filter=None               ->  iitpkd_lift p2, niti_sharp_tender p3, nitt_glove_box p3
filter=iitpkd_lift        ->  iitpkd_lift p1, iitpkd_lift p2, iitpkd_lift p20
filter=nitt_glove_box     ->  nitt_glove_box p2, nitt_glove_box p3


---
## Step 5: build the prompt

This is the "Augmented" part. On its own the model has never seen these tenders,
so it would either refuse or invent an answer. Here we paste the retrieved
clauses into the prompt as `{context}` and tell the model to answer only from
them.

Two instructions in the template matter a lot:

- answer only from the context, so the model cannot fall back on general
  knowledge and make up an EMD figure
- if the context does not contain the answer, say so. For a tender, a confident
  wrong number is much worse than "this is not stated".

In [13]:
from rag.step5_prompt import TEMPLATE

print(TEMPLATE)

You are helping a bidder read a government tender document.

Answer the question using ONLY the tender extracts given below.

Rules:
- If the extracts do not contain the answer, reply exactly:
  "This is not stated in the tender extracts I was given."
- Do not use any knowledge from outside the extracts.
- Quote the figure or the wording from the tender where you can.
- Keep the answer short, at most 4 sentences.

Tender extracts:
{context}

Question: {question}

Answer:


### The exact text that gets sent to the model

This is the single most useful cell in the notebook. It prints the finished
prompt, with the retrieved tender clauses sitting inside it. You can read the
context and then read the answer and check that one came from the other.

In [14]:
from rag.step5_prompt import build_prompt

chunks_for_prompt = retrieve_chunks(question, k=3, tender="iitpkd_lift")

print(build_prompt(question, chunks_for_prompt))

Human: You are helping a bidder read a government tender document.

Answer the question using ONLY the tender extracts given below.

Rules:
- If the extracts do not contain the answer, reply exactly:
  "This is not stated in the tender extracts I was given."
- Do not use any knowledge from outside the extracts.
- Quote the figure or the wording from the tender where you can.
- Keep the answer short, at most 4 sentences.

Tender extracts:
[Extract 1 | iitpkd_lift | page 2]
or explanation by any other employee of any of the Sections/Departments of the Institute, shall be taken to bind or fetter the Institute. 1.11. EARNEST MONEY DEPOSIT (EMD) 1.12. The bidder shall furnish EMD of Rs.56,000/- (Rupees Fifty Six Thousand Only) through an online payment gateway in the

[Extract 2 | iitpkd_lift | page 1]
3 Date / Time of closing 11-05-2026, 1100 hrs 4 Opening of Bids 11-05-2026, 1130 hrs 5 Estimated Cost put to bid Rs. 28,00,000/- 6 Earnest Money Deposit (EMD) Rs. 56,000/- Engineering works D

---
## Step 6: generate the answer

Two ways to call the model live in `step6_generate.py`, on purpose.

**The class way** - LangChain's `init_chat_model` with an LCEL chain:

```python
chain = RAG_PROMPT | llm | StrOutputParser()
```

**The resilience way** - LiteLLM, which tries a primary model and falls back to
a second model if the first call raises.

In [15]:
from rag.step6_generate import CHAT_MODEL, TEMPERATURE, generate_answer

print("model:", CHAT_MODEL, "| temperature:", TEMPERATURE)
print()

answer, used_chunks = generate_answer(question, tender="iitpkd_lift")

print("Question:", question)
print("Answer  :", answer)
print()
print("Sources:")
for source in sorted({f"{c.metadata['tender']}, page {c.metadata['page']}" for c in used_chunks}):
    print("  -", source)

model: openai/gpt-oss-120b | temperature: 0.0



Question: What is the earnest money deposit for the home lift?
Answer  : The tender specifies an Earnest Money Deposit of **Rs. 56,000/- (Rupees Fifty‑Six Thousand Only)**【Extract 1】【Extract 2】.

Sources:
  - iitpkd_lift, page 1
  - iitpkd_lift, page 10
  - iitpkd_lift, page 2
  - iitpkd_lift, page 20
  - iitpkd_lift, page 9


Temperature is 0 because reading a tender is a factual task, not a creative one.
We want the same question to give the same answer every time. The class notebook
on generation parameters puts deterministic, extraction-style tasks at the low
end of the range.

### A few more questions

In [16]:
questions = [
    ("What is the estimated cost put to bid?", "iitpkd_lift"),
    ("What is the last date and time for closing of bids?", "iitpkd_lift"),
    ("What is the EMD for the glove box tender?", "nitt_glove_box"),
    ("What is the bid validity period?", "niti_sharp_tender"),
]

for q, t in questions:
    a, _ = generate_answer(q, tender=t)
    print(f"Q: {q}  [{t}]")
    print(f"A: {a}")
    print()

Q: What is the estimated cost put to bid?  [iitpkd_lift]
A: The estimated cost put to bid is **Rs. 28,00,000/-**【Extract 4 | iitpkd_lift | page 1】.



Q: What is the last date and time for closing of bids?  [iitpkd_lift]
A: The last date and time for closing of bids is **11‑05‑2026, 1100 hrs**【Extract 4 | page 1】.



Q: What is the EMD for the glove box tender?  [nitt_glove_box]
A: The Earnest Money Deposit (EMD) required is **Rs 6,000 /- (Rupees Six thousand only)**【Extract 7 | nitt_glove_box | page 2】.



Q: What is the bid validity period?  [niti_sharp_tender]
A: The bid validity period is **75 days** from the last date of submission of the bids (i.e., 1 Sept 2020)【Extract 1 | niti_sharp_tender | page 10】.



### What happens when the answer is not in the document?

This matters more than it looks. A bidder acting on an invented EMD figure loses
the bid. The prompt tells the model to refuse rather than guess, and here we
check that it actually does.

In [17]:
out_of_context = [
    "Who is the CEO of Microsoft?",
    "What is the penalty for late delivery of a passport?",
]

for q in out_of_context:
    a, _ = generate_answer(q)
    print(f"Q: {q}")
    print(f"A: {a}")
    print()

Q: Who is the CEO of Microsoft?
A: This is not stated in the tender extracts I was given.



Q: What is the penalty for late delivery of a passport?
A: This is not stated in the tender extracts I was given.



---
## The LiteLLM fallback

Our teacher liked the fallback idea in our previous project, so we kept it, but
made it small enough to read in one go. It is one function in
`rag/step6_generate.py`:

```python
try:
    response = litellm.completion(model=PRIMARY_MODEL, messages=messages, ...)
    return response.choices[0].message.content, PRIMARY_MODEL
except Exception as error:
    print("primary model failed, switching to", FALLBACK_MODEL)
    response = litellm.completion(model=FALLBACK_MODEL, messages=messages, ...)
    return response.choices[0].message.content, FALLBACK_MODEL
```

The practical reason is that a free API tier can rate-limit in the middle of a
demo. The same prompt simply goes to a second model instead.

In [18]:
import rag.step6_generate as generate

print("=== normal run ===")
a, _, model_used = generate.answer_with_fallback(question, tender="iitpkd_lift")
print("answered by:", model_used)
print(a)

=== normal run ===


answered by: groq/openai/gpt-oss-120b
The Earnest Money Deposit (EMD) is **Rs. 56,000/- (Rupees Fifty‑Six Thousand Only)**【Extract 1】【Extract 2】.


In [19]:
print("=== now we break the primary model on purpose ===")

real_primary = generate.PRIMARY_MODEL
generate.PRIMARY_MODEL = "groq/this-model-does-not-exist"

a, _, model_used = generate.answer_with_fallback(question, tender="iitpkd_lift")
print("answered by:", model_used)
print(a)

generate.PRIMARY_MODEL = real_primary   # put it back

=== now we break the primary model on purpose ===

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

  primary model failed (NotFoundError), switching to groq/openai/gpt-oss-20b


answered by: groq/openai/gpt-oss-20b
The earnest money deposit (EMD) for the home lift is **Rs. 56,000/- (Rupees Fifty‑Six Thousand Only)**【Extract 1】【Extract 2】.


The primary call raises, the exception is caught, the same prompt goes to the
fallback model, and the answer still comes back - with the app reporting which
model produced it.

---
## What we learned

- Chunk size mattered far more than we expected. Going from 1000 to 300
  characters took our hand-checked test from 1 of 6 to 5 of 6.
- Retrieval is the part that decides whether the answer is right. When the right
  chunk is not retrieved, a good model still cannot answer.
- Telling the model to refuse when the context does not cover the question is
  what makes the output trustworthy for a tender.
- Metadata filtering was necessary as soon as we indexed more than one tender.

## Limitations

- One of our three tenders is a scanned document, and its OCR text has errors
  ("75 davs" instead of "75 days"). That noise hurts retrieval on that document.
- We judged retrieval with six questions we checked by hand. That is a sanity
  check, not a proper evaluation with a labelled test set.
- Only PDFs with a real text layer work. A purely scanned PDF would need OCR,
  which we did not add.
- The answers are only as good as the retrieved chunks. If the right clause is
  not in the top k, the system says it does not know rather than finding it.